<a href="https://colab.research.google.com/github/Bishal7474/CSCE1015/blob/main/Copy_of_Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q smolagents ddgs requests

In [ ]:
from getpass import getpass

hf_token = getpass("Paste your NEW Hugging Face token: ").strip()
print("Token entered" if hf_token else "No token entered")

Paste your NEW Hugging Face token: ··········
Token entered


In [ ]:
%pip install -q huggingface_hub ddgs requests beautifulsoup4 openpyxl pandas faster-whisper youtube-transcript-api

import base64
import io
import json
import mimetypes
import re
import subprocess
import sys
import tempfile
import time
from getpass import getpass
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
from faster_whisper import WhisperModel
from huggingface_hub import InferenceClient

# ----------------------------------------------------------------- config
BASE_URL = "https://agents-course-unit4-scoring.hf.space"
MODEL = "openai/gpt-oss-120b"
VISION_MODEL = "Qwen/Qwen2.5-VL-72B-Instruct"   # used only to look at images
MAX_STEPS = 12
FILES_DIR = Path("/content/gaia_files")
SAVE_FILE = Path("/content/gaia_answers_v4.json")
OLD_FILES = [Path("/content/gaia_answers_v2.json"), Path("/content/gaia_answers_v3.json")]

# Question numbers (1-based, as printed) to solve again even if an answer is saved.
# Use this after you see which ones were scored wrong. Use "all" to redo everything.
FORCE_REDO = set()          # e.g. {3, 7, 12}
REDO_ALL = False

# Fill these in to submit at the end.
USERNAME = ""               # your Hugging Face username
AGENT_CODE_URL = ""         # e.g. https://huggingface.co/spaces/<you>/<space>/tree/main
SUBMIT = False

token = globals().get("hf_token") or getpass("Paste your HF inference token: ").strip()
client = InferenceClient(provider="auto", token=token)

tasks = requests.get(f"{BASE_URL}/questions", timeout=30).json()
task_ids = {t["task_id"] for t in tasks}
number_of = {t["task_id"]: i for i, t in enumerate(tasks, start=1)}

# ------------------------------------------------------------ saved state
saved = {}
for path in OLD_FILES + [SAVE_FILE]:
    if path.exists():
        for row in json.loads(path.read_text()):
            if row["task_id"] in task_ids:
                saved[row["task_id"]] = row["submitted_answer"]

answers = []


def save_answers():
    answers[:] = [{"task_id": k, "submitted_answer": v} for k, v in saved.items()]
    tmp = SAVE_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(answers, ensure_ascii=False, indent=2))
    tmp.replace(SAVE_FILE)


save_answers()
print(f"Starting with {len(answers)} saved answers.", flush=True)


# -------------------------------------------------------------------- LLM
def chat(messages, max_tokens=4000):
    # gpt-oss is a reasoning model: a small max_tokens can leave the visible reply empty.
    for attempt in range(4):
        try:
            r = client.chat_completion(model=MODEL, messages=messages, max_tokens=max_tokens)
            text = (r.choices[0].message.content or "").strip()
            if text:
                return text
        except Exception:
            if attempt == 3:
                raise
        time.sleep(5 * (attempt + 1))
    return ""


# ------------------------------------------------------------ attachments
speech_model = None


def download(task):
    FILES_DIR.mkdir(exist_ok=True)
    path = FILES_DIR / f"{task['task_id']}_{task['file_name']}"
    if not path.exists():
        r = requests.get(f"{BASE_URL}/files/{task['task_id']}", timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path


def describe_attachment(task):
    global speech_model
    path = download(task)
    suffix = path.suffix.lower()

    if suffix == ".mp3":
        if speech_model is None:
            print("  Loading speech model...", flush=True)
            speech_model = WhisperModel("small", device="cpu", compute_type="int8")
        segments, _ = speech_model.transcribe(str(path), beam_size=5)
        text = " ".join(s.text.strip() for s in segments)
        return f"File: {path}\nAudio transcript:\n{text}"

    if suffix in {".png", ".jpg", ".jpeg", ".gif", ".webp"}:
        return f"Image file at {path}. Use the look tool to inspect it."

    if suffix in {".xlsx", ".xls", ".csv", ".tsv"}:
        import pandas as pd
        if suffix in {".xlsx", ".xls"}:
            sheets = pd.read_excel(path, sheet_name=None)
        else:
            sheets = {"data": pd.read_csv(path, sep="\t" if suffix == ".tsv" else ",")}
        preview = "\n\n".join(
            f"Sheet '{n}' shape={df.shape}\n{df.head(15).to_string()}" for n, df in sheets.items()
        )
        return (f"Spreadsheet at {path}. Preview:\n{preview}\n"
                "Compute exact answers with the python tool and pandas; do not add by hand.")

    if suffix == ".py":
        code = path.read_text(errors="replace")
        return (f"Python file at {path}. Source:\n{code}\n"
                "Run it with the python tool (subprocess on that path) to get the real output.")

    if suffix == ".pdf":
        return f"PDF at {path}. Extract its text with the python tool."

    return f"File at {path}:\n" + path.read_text(errors="replace")[:20000]


# ------------------------------------------------------------------ tools
def tool_search(query):
    hits = []
    for backend in ("auto", "duckduckgo", "yahoo", "bing"):
        try:
            hits = DDGS(timeout=15).text(query, backend=backend, max_results=8)
            if hits:
                break
        except Exception:
            continue
    return "\n\n".join(f"{h.get('title','')}\n{h.get('href','')}\n{h.get('body','')}"
                       for h in hits) or "No results. Try different wording."


def tool_fetch(url, find="", offset=0):
    r = requests.get(url, timeout=25, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    if "html" not in r.headers.get("content-type", "").lower():
        return "Not an HTML page. If it is a PDF or data file, download it in the python tool."
    soup = BeautifulSoup(r.text, "html.parser")
    for el in soup(["script", "style", "nav", "footer", "sup"]):
        el.decompose()
    for row in soup.find_all("tr"):          # keep table rows on one line
        try:
            cells = [c.get_text(" ", strip=True) for c in row.find_all(["th", "td"])]
            row.replace_with(soup.new_string(" | ".join(cells) + "\n"))
        except Exception:
            pass
    text = re.sub(r"\n\s*\n+", "\n", soup.get_text("\n"))

    if find:
        spans = [m.start() for m in re.finditer(re.escape(find), text, flags=re.I)]
        if not spans:
            return f"'{find}' not found on the page (length {len(text)})."
        out, last = [], -1
        for s in spans[:12]:
            if s < last:
                continue
            out.append(text[max(0, s - 700): s + 1200])
            last = s + 1200
        return "\n-----\n".join(out)[:9000]

    chunk = text[offset: offset + 8000]
    return f"[chars {offset}-{offset + len(chunk)} of {len(text)}]\n{chunk}"


def tool_python(code):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
    try:
        p = subprocess.run([sys.executable, f.name], capture_output=True, text=True, timeout=90)
        return ((p.stdout or "") + (p.stderr or ""))[-5000:] or "(no output)"
    except subprocess.TimeoutExpired:
        return "Timed out after 90 seconds."


def tool_youtube(url):
    m = re.search(r"(?:v=|youtu\.be/)([\w-]{11})", url)
    if not m:
        return "Could not find a video id in that URL."
    vid = m.group(1)
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        try:
            data = YouTubeTranscriptApi().fetch(vid)
            return " ".join(s.text for s in data)[:9000]
        except AttributeError:
            data = YouTubeTranscriptApi.get_transcript(vid)
            return " ".join(s["text"] for s in data)[:9000]
    except Exception as e:
        return (f"No transcript available ({type(e).__name__}). "
                "If the question needs what is SHOWN in the video, it cannot be answered from text.")


def tool_look(path, question):
    mime = mimetypes.guess_type(path)[0] or "image/png"
    b64 = base64.b64encode(Path(path).read_bytes()).decode()
    r = client.chat_completion(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
            {"type": "text", "text": question},
        ]}],
        max_tokens=1200,
    )
    return r.choices[0].message.content or "(empty)"


def run_tool(action):
    name = action.get("action")
    try:
        if name == "search":
            return tool_search(action["query"])
        if name == "fetch":
            return tool_fetch(action["url"], action.get("find", ""), int(action.get("offset", 0)))
        if name == "python":
            return tool_python(action["code"])
        if name == "youtube":
            return tool_youtube(action["url"])
        if name == "look":
            return tool_look(action["path"], action["question"])
        return f"Unknown action '{name}'."
    except Exception as e:
        return f"Tool error: {type(e).__name__}: {str(e)[:300]}"


# ------------------------------------------------------------------ agent
SYSTEM = """You are a careful research agent answering GAIA benchmark questions.
Each turn, reply with exactly ONE JSON object and nothing else. Actions:
{"action":"search","query":"..."}                      web search (titles, urls, snippets)
{"action":"fetch","url":"...","find":"word","offset":0} read a page; "find" jumps to matching passages
{"action":"python","code":"..."}                       run Python (pandas, requests available); print() results
{"action":"youtube","url":"..."}                       get a video's transcript
{"action":"look","path":"...","question":"..."}        ask a vision model about an image file
{"action":"final","answer":"..."}                      submit the final answer

Rules:
- Use Wikipedia, official sites and primary sources. Search snippets are not enough for counts, dates
  or table values: fetch the page and read the exact rows. Re-check the exact wording of the question
  (date ranges, "between", inclusive/exclusive, which edition, which year).
- Use python for any arithmetic, counting, sorting, spreadsheets, or code behavior. Never do long
  calculations by hand. Many riddle-style questions (reversed text, logic tables, word puzzles)
  should be solved by reasoning carefully, or checked with python.
- If one source fails, try another query or another source. Do not give up early.
- The final answer is graded by EXACT MATCH. Give only the answer: no sentence, no explanation.
  Numbers: digits only, no commas or units unless asked. Text: no leading article, no trailing period.
  Lists: comma-separated, in the order/format the question asks (e.g. alphabetical if requested).
  Follow any format instruction in the question literally (rounding, abbreviations, names format).
- Always end with a best-guess final answer, never UNKNOWN."""


def parse_action(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def clean(answer):
    answer = str(answer).strip().strip("`").strip()
    answer = re.sub(r"^(final answer|answer)\s*[:\-]\s*", "", answer, flags=re.I)
    if answer.endswith(".") and not re.search(r"\d\.$", answer):
        answer = answer[:-1]
    return answer.strip()


def solve(task):
    context = describe_attachment(task) if task.get("file_name") else ""
    user = f"Question:\n{task['question']}\n"
    if context:
        user += f"\nAttachment:\n{context[:30000]}\n"
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

    for step in range(MAX_STEPS):
        reply = chat(messages)
        messages.append({"role": "assistant", "content": reply})
        action = parse_action(reply)
        if not action:
            messages.append({"role": "user", "content": "Reply with exactly one JSON object."})
            continue
        if action.get("action") == "final":
            return clean(action.get("answer", ""))
        observation = run_tool(action)
        print(f"    step {step + 1}: {action.get('action')}", flush=True)
        messages.append({"role": "user", "content": "Observation:\n" + observation[:9000]})

    messages.append({"role": "user", "content":
                     'Out of steps. Reply now with {"action":"final","answer":"<best guess>"}.'})
    action = parse_action(chat(messages)) or {}
    return clean(action.get("answer", ""))


# -------------------------------------------------------------------- run
for task in tasks:
    task_id, number = task["task_id"], number_of[task["task_id"]]
    redo = REDO_ALL or number in FORCE_REDO
    if task_id in saved and not redo:
        continue

    print(f"\nQuestion {number} of {len(tasks)}: {task['question'][:90]}...", flush=True)
    try:
        answer = solve(task)
    except Exception as error:
        print("  Skipped:", str(error).replace(token, "[hidden]")[:350])
        continue

    if not answer:
        print("  No answer produced.")
        continue

    saved[task_id] = answer
    save_answers()
    print("  ANSWER:", answer, flush=True)

print(f"\nCompleted answers: {len(answers)} of {len(tasks)}.")

# ----------------------------------------------------------------- submit
if SUBMIT and USERNAME and AGENT_CODE_URL:
    result = requests.post(
        f"{BASE_URL}/submit",
        json={"username": USERNAME, "agent_code": AGENT_CODE_URL, "answers": answers},
        timeout=90,
    )
    print(result.status_code, result.text)
else:
    print("Set USERNAME, AGENT_CODE_URL and SUBMIT = True to submit.")

Starting with 9 saved answers.

Question 2 of 20: In the video https://www.youtube.com/watch?v=L1vXCYZAYYM, what is the highest number of bi...
    step 1: youtube
    step 2: fetch
    step 4: search
    step 6: search


    step 7: search
    step 9: fetch


    step 10: search
  ANSWER: 5

Question 4 of 20: Review the chess position provided in the image. It is black's turn. Provide the correct n...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/cca530fc-4052-43b2-b130-b30968d8aa44

Question 5 of 20: Who nominated the only Featured Article on English Wikipedia about a dinosaur that was pro...
    step 1: search
    step 2: None
  ANSWER: User:Aron

Question 10 of 20: Hi, I'm making a pie but I could use some help with my shopping list. I have everything I ...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3

Question 12 of 20: What is the final numeric output from the attached Python code?...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/f918266a-b3e0-4914-865d-4faa564f1aef

Question 14 of 20: Hi, I was out sick from my classes on Friday, so I'm tryi

    step 1: search
    step 3: None
    step 4: None
    step 5: None
    step 7: None
    step 8: fetch
    step 9: None
    step 10: None
    step 11: None
    step 12: None
  ANSWER: Saint Petersburg

Question 17 of 20: What country had the least number of athletes at the 1928 Summer Olympics? If there's a ti...
    step 3: search
    step 4: fetch
    step 6: python
    step 7: search
    step 9: None
    step 11: fetch
    step 12: fetch
  ANSWER: CUB

Question 19 of 20: The attached Excel file contains the sales of menu items for a local fast-food chain. What...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/7bd855d8-463d-4ed5-93ca-5fe35145f733

Question 20 of 20: What is the first name of the only Malko Competition recipient from the 20th Century (afte...
    step 1: search
    step 6: fetch
    step 7: fetch
    step 8: search
    step 9: fetch
    step 10: search
    step 12: fetch
  ANSWER: Gotthard

Completed answers: 14 of

In [ ]:
%pip install -q huggingface_hub ddgs requests beautifulsoup4 openpyxl faster-whisper

import io
import json
import tempfile
from getpass import getpass
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
from faster_whisper import WhisperModel
from huggingface_hub import InferenceClient
from openpyxl import load_workbook

BASE_URL = "https://agents-course-unit4-scoring.hf.space"
MODEL = "openai/gpt-oss-120b"
SAVE_FILE = Path("/content/gaia_answers_v3.json")
OLD_FILE = Path("/content/gaia_answers_v2.json")

# Reuse the working token if this Colab session still has it.
token = globals().get("hf_token") or getpass("Paste your HF inference token: ").strip()
client = InferenceClient(provider="auto", token=token)

response = requests.get(f"{BASE_URL}/questions", timeout=30)
response.raise_for_status()
tasks = response.json()
task_ids = {task["task_id"] for task in tasks}

# Keep previous answers, including the seven from your last run.
saved = {}
for path in (OLD_FILE, SAVE_FILE):
    if path.exists():
        for row in json.loads(path.read_text()):
            if row["task_id"] in task_ids:
                saved[row["task_id"]] = row["submitted_answer"]

for row in globals().get("answers", []) or []:
    if row["task_id"] in task_ids:
        saved.setdefault(row["task_id"], row["submitted_answer"])

answers = []
def save_answers():
    answers[:] = [
        {"task_id": task_id, "submitted_answer": answer}
        for task_id, answer in saved.items()
    ]
    temporary = SAVE_FILE.with_suffix(".tmp")
    temporary.write_text(json.dumps(answers, ensure_ascii=False, indent=2))
    temporary.replace(SAVE_FILE)

save_answers()
print(f"Starting with {len(answers)} saved answers.", flush=True)


def ask_model(prompt, max_tokens=1800):
    result = client.chat_completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return (result.choices[0].message.content or "").strip()


speech_model = None

def read_attachment(task):
    global speech_model
    suffix = Path(task["file_name"]).suffix.lower()

    if suffix not in {".py", ".txt", ".csv", ".tsv", ".json", ".md", ".xlsx", ".mp3"}:
        raise ValueError(f"Attachment type {suffix} needs another reader.")

    response = requests.get(f"{BASE_URL}/files/{task['task_id']}", timeout=45)
    response.raise_for_status()
    data = response.content

    if suffix == ".mp3":
        if speech_model is None:
            print("  Loading speech recognition model...", flush=True)
            speech_model = WhisperModel("base", device="cpu", compute_type="int8")
        with tempfile.NamedTemporaryFile(suffix=".mp3") as audio:
            audio.write(data)
            audio.flush()
            segments, _ = speech_model.transcribe(audio.name, beam_size=5)
            return "Audio transcript:\n" + " ".join(s.text.strip() for s in segments)

    if suffix == ".xlsx":
        workbook = load_workbook(io.BytesIO(data), read_only=True, data_only=True)
        try:
            lines = []
            for sheet in workbook:
                lines.append(f"Sheet: {sheet.title}")
                for row in sheet.iter_rows(values_only=True):
                    lines.append("\t".join("" if v is None else str(v) for v in row))
                    if len(lines) >= 500:
                        return "\n".join(lines)
            return "\n".join(lines)
        finally:
            workbook.close()

    # Python attachments are read as source text. Nothing in them is executed.
    return data.decode("utf-8-sig", errors="replace")


def search(query):
    for backend in ("yahoo", "auto"):
        try:
            hits = DDGS(timeout=12).text(query, backend=backend, max_results=4)
            if hits:
                return hits
        except Exception:
            continue
    return []


def read_page(url, query):
    try:
        with requests.get(url, timeout=10, stream=True) as page:
            page.raise_for_status()
            if "html" not in page.headers.get("content-type", "").lower():
                return ""
            data = bytearray()
            for chunk in page.iter_content(16384):
                data.extend(chunk)
                if len(data) >= 600_000:
                    break
        soup = BeautifulSoup(data, "html.parser")
        for element in soup(["script", "style", "nav", "footer"]):
            element.decompose()
        lines = [" ".join(line.split()) for line in soup.stripped_strings]
        terms = {word.lower().strip(".,?!\"'") for word in query.split() if len(word) > 3}
        # Keep neighboring lines so table entries retain names, years and values.
        relevant = set()
        for index, line in enumerate(lines):
            if any(term in line.lower() for term in terms):
                relevant.update(range(max(0, index - 2), min(len(lines), index + 4)))
        excerpt = "\n".join(lines[i] for i in sorted(relevant))
        return (excerpt or "\n".join(lines))[:6000]
    except (requests.RequestException, ValueError):
        return ""


def research(question):
    query = ask_model(
        "Write one short web search query for the question below. "
        "Return only the query.\n\n" + question
    )
    evidence = []
    seen_urls = set()

    for round_number in range(2):
        if not query:
            break
        for hit in search(query):
            url = hit.get("href", "")
            if url in seen_urls or not url.startswith(("https://", "http://")):
                continue
            seen_urls.add(url)
            snippet = f"{hit.get('title', '')}\n{hit.get('body', '')}"
            excerpt = read_page(url, query)
            evidence.append(f"{url}\n{snippet}\n{excerpt}")

        if round_number == 0:
            query = ask_model(
                f"Question: {question}\n\nEvidence:\n"
                + "\n\n".join(evidence)[:16000]
                + "\n\nIf there is enough evidence, reply exactly DONE. "
                  "Otherwise give one focused search query for the missing fact."
            )
            if query.strip().upper().rstrip(".") == "DONE":
                break

    return "\n\n".join(evidence)[:32000]


def solve(task):
    question = task["question"]
    if task.get("file_name"):
        evidence = read_attachment(task)[:40000]
    else:
        evidence = research(question)

    return ask_model(
        f"Question:\n{question}\n\nSource material:\n{evidence}\n\n"
        "Use the source material as evidence. For a program, trace its behavior "
        "carefully. For a transcript, check the exact words and order requested. "
        "You may solve self-contained reasoning questions directly. "
        "Return only the final answer in the exact format the question requests. "
        "If essential evidence is missing, return exactly UNKNOWN."
    )


# Handle previously skipped attachments first.
ordered_tasks = sorted(
    enumerate(tasks, start=1),
    key=lambda item: not bool(item[1].get("file_name")),
)

for number, task in ordered_tasks:
    task_id = task["task_id"]
    if task_id in saved:
        continue

    print(f"\nTrying question {number} of {len(tasks)}...", flush=True)
    try:
        answer = solve(task)
    except Exception as error:
        print("  Skipped:", str(error).replace(token, "[hidden]")[:350])
        continue

    if not answer or answer.upper().rstrip(".") == "UNKNOWN":
        print("  Could not verify an answer.")
        continue

    saved[task_id] = answer
    save_answers()
    print("  ANSWER:", answer, flush=True)

print(f"\nCompleted answers: {len(answers)} of {len(tasks)}.")

Starting with 7 saved answers.

Trying question 4 of 20...
  Skipped: Attachment type .png needs another reader.

Trying question 10 of 20...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3

Trying question 12 of 20...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/f918266a-b3e0-4914-865d-4faa564f1aef

Trying question 14 of 20...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/1f975693-876d-457b-a649-393859e79bf3

Trying question 19 of 20...
  Skipped: 404 Client Error: Not Found for url: https://agents-course-unit4-scoring.hf.space/files/7bd855d8-463d-4ed5-93ca-5fe35145f733

Trying question 1 of 20...
  ANSWER: We can try to use browser tool.Search.We need to retrieve the discography section.The tool not responding, maybe need to use proper browser.open?Probably the environment not set. However, m

In [ ]:
import requests

saved_answers = globals().get("answers")

if not saved_answers:
    print("No saved answers. Rerun your evaluation cell first.")
else:
    print(f"Ready to submit {len(saved_answers)} answers as AgentsBishal.")

    if input("Type SUBMIT to continue: ").strip() == "SUBMIT":
        response = requests.post(
            "https://agents-course-unit4-scoring.hf.space/submit",
            json={
                "username": "AgentsBishal",
                "agent_code": (
                    "https://huggingface.co/spaces/AgentsBishal/"
                    "Final_Assignment_Template/tree/main"
                ),
                "answers": saved_answers,
            },
            timeout=60,
        )
        print("HTTP status:", response.status_code)
        print(response.text)

Ready to submit 14 answers as AgentsBishal.
Type SUBMIT to continue: SUBMIT
HTTP status: 200
{"username":"AgentsBishal","score":30.0,"correct_count":6,"total_attempted":14,"message":"Score calculated successfully: 6/20 total questions answered correctly (14 valid tasks attempted). High score updated on leaderboard.","timestamp":"2026-11-19T00:12:04.651460+00:00"}


In [ ]:
from ddgs import DDGS

query = "Malko Competition winners"

for backend in ("brave", "google", "bing"):
    try:
        results = DDGS().text(query, backend=backend, max_results=3)
        if results:
            print("WORKING SEARCH:", backend)
            for result in results:
                print(result["title"], "-", result["href"])
            break
        print(f"{backend}: no results")
    except Exception as error:
        print(f"{backend}: {type(error).__name__}")
else:
    print("No search backend returned results.")

brave: DDGSException


google: DDGSException
WORKING SEARCH: bing
Malko Competition - Wikipedia - https://en.wikipedia.org/wiki/Malko_Competition
馬爾科國際青年指揮大賽 - 维基百科，自由的百科全书 - https://zh.wikipedia.org/wiki/馬爾科國際青年指揮大賽
All Winners | Malko Competition - https://www.malkocompetition.dk/winners/all
